In [19]:
!pip install -q langchain==0.2.0 langchain-openai==0.1.8 langchain-community==0.2.0 langchain-chroma
!pip install -q openai==1.30.0 httpx==0.27.2 pypdf python-dotenv

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain_openai import OpenAIEmbeddings
import httpx

http_client = httpx.Client()
http_async_client = httpx.AsyncClient()

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    http_client=http_client,
    http_async_client=http_async_client,
    check_embedding_ctx_length=False,
    chunk_size=40,
    max_retries=2,
    request_timeout=60,
 )

In [23]:
import pathlib
from langchain_community.document_loaders import PyPDFLoader

paths = list(pathlib.Path("./data").glob("**/*.pdf"))

docs = []
for path in paths:
    loader = PyPDFLoader(str(path))
    docs.extend(loader.load())

print(len(docs))

8


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

chunks = splitter.split_documents(docs)
print(len(chunks))

55


In [25]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_collection",
    persist_directory="./chroma_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [27]:
def retrieve(query, k=3):
    results = db.similarity_search_with_score(query, k=k)
    return results

In [29]:
from langchain_openai import ChatOpenAI
import httpx

http_client = httpx.Client()
http_async_client = httpx.AsyncClient()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    http_client=http_client,
    http_async_client=http_async_client,
    request_timeout=60,
    max_retries=2,
)

In [30]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Use ONLY the context below to answer.
If the context does not contain the answer, say: "I don't know".

Context:
{context}

Question:
{question}
""",
    input_variables=["context", "question"]
)

In [31]:
def rag_agent(query):
    
    results = retrieve(query, k=3)

    best_score = results[0][1] if results else 1.0

    # ❌ Case 3: No good match in RAG
    if best_score > 0.35:
        return "I don't know (no relevant documents found)"

    context = "\n".join([r[0].page_content for r in results])

    response = llm.invoke(
        rag_prompt.format(context=context, question=query)
    )

    return response.content

In [32]:
print(rag_agent("what is document?"))

I don't know (no relevant documents found)


In [33]:
from langchain_core.prompts import PromptTemplate

# Define the agent's specialty domain
AGENT_SPECIALTY = "board games (Monopoly, Ticket to Ride)"
SIMILARITY_THRESHOLD = 2.0  # Threshold for considering a document as relevant (Euclidean distance)

# Create RAG prompt template
rag_prompt = PromptTemplate(
    template="""You are an expert assistant specializing in: {specialty}

Use ONLY the provided context to answer the question. 

Context:
{context}

Question: {question}

Important instructions:
- If the context contains relevant information, provide a helpful answer based on it.
- If the context does NOT contain relevant information about the question, respond with: "I don't know."
- Be concise and clear in your response.

Answer:""",
    input_variables=['specialty', 'context', 'question']
)

print("✓ RAG prompt template created")
print(f"✓ Agent specialty: {AGENT_SPECIALTY}")
print(f"✓ Similarity threshold: {SIMILARITY_THRESHOLD} (Euclidean distance)")

✓ RAG prompt template created
✓ Agent specialty: board games (Monopoly, Ticket to Ride)
✓ Similarity threshold: 2.0 (Euclidean distance)


In [34]:
def rag_agent(user_question: str) -> dict:
    """
    RAG agent that handles 3 cases:
    1. Topic outside agent specialty but found in RAG → return answer
    2. Topic in agent specialty and found in RAG → return answer
    3. Topic in agent specialty but NOT found in RAG → return "I don't know"
    """
    # Search for similar documents
    search_results = db.similarity_search_with_score(user_question, k=3)
    
    if not search_results:
        return {
            "question": user_question,
            "answer": "I don't know.",
            "source": "Knowledge base",
            "confidence": "Low - No documents found",
            "case": "Case 3: Topic in specialty but not found in RAG"
        }
    
    # Get the best match score (lower = more similar for L2 distance)
    best_doc, best_score = search_results[0]
    
    # If best match is too dissimilar (distance too high)
    if best_score > SIMILARITY_THRESHOLD:
        return {
            "question": user_question,
            "answer": "I don't know.",
            "source": "Knowledge base",
            "confidence": f"Low - Best match distance: {best_score:.3f} (threshold: {SIMILARITY_THRESHOLD})",
            "case": "Case 3: Topic in specialty but not found in RAG"
        }
    
    # Format context from top documents
    context = "\n---\n".join([doc.page_content for doc, _ in search_results[:3]])
    
    # Build the prompt
    prompt_text = rag_prompt.format(
        specialty=AGENT_SPECIALTY,
        context=context,
        question=user_question
    )
    
    # Get answer from LLM
    response = llm.invoke(prompt_text)
    answer = response.content
    
    return {
        "question": user_question,
        "answer": answer,
        "source": "RAG (Chroma Vector DB)",
        "confidence": f"High - Best match distance: {best_score:.3f}",
        "case": "Case 1/2: Topic found in RAG → Returned answer",
        "documents_used": len(search_results)
    }

print("✓ RAG Agent function updated with corrected threshold")

✓ RAG Agent function updated with corrected threshold


In [35]:
from pprint import pprint

# Test the RAG agent with different queries
test_queries = [
    "How do I get out of jail in Monopoly?",
    "What are the rules of Ticket to Ride?",
    "How do I cook spaghetti?"  # Topic outside specialty
]

print("=" * 80)
print("RAG AGENT TEST - Demonstrating 3 Cases")
print("=" * 80)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}]: {query}")
    result = rag_agent(query)
    print(f"\n{result['case']}")
    print(f"Answer: {result['answer']}")
    print(f"Confidence: {result['confidence']}")
    print("-" * 80)

RAG AGENT TEST - Demonstrating 3 Cases

[Query 1]: How do I get out of jail in Monopoly?

Case 1/2: Topic found in RAG → Returned answer
Answer: To get out of jail in Monopoly, you can do one of the following:

1. Pay a fine of £50 and continue on your next turn.
2. Use a "Get Out Of Jail Free" card if you have one.
3. Purchase a "Get Out Of Jail Free" card from another player at a mutually agreed price.
4. Wait for three turns, rolling the dice on each turn to try to roll a double. If you roll a double on any turn, you can move out of jail using that dice roll.
Confidence: High - Best match distance: 0.484
--------------------------------------------------------------------------------

[Query 2]: What are the rules of Ticket to Ride?

Case 1/2: Topic found in RAG → Returned answer
Answer: I don't know.
Confidence: High - Best match distance: 1.074
--------------------------------------------------------------------------------

[Query 3]: How do I cook spaghetti?

Case 1/2: Topic fou

In [37]:
def interactive_rag():
    """
    Interactive RAG interface for user queries
    """
    print("\n" + "=" * 80)
    print("INTERACTIVE RAG AGENT - Ask About Board Games!")
    print("=" * 80)
    print(f"Agent Specialty: {AGENT_SPECIALTY}")
    print("Available documents: Monopoly rules, Ticket to Ride rules")
    print("\nEnter your question (or 'quit' to exit):")
    print("=" * 80 + "\n")
    
    while True:
        user_input = input("Your question: ").strip()
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n✓ Goodbye!")
            break
        
        if not user_input:
            print("Please enter a valid question.\n")
            continue
        
        print("\n⏳ Processing...")
        result = rag_agent(user_input)
        
        print(f"\n{'─' * 80}")
        print(f"Case: {result['case']}")
        print(f"Answer: {result['answer']}")
        print(f"Confidence: {result['confidence']}")
        print(f"{'─' * 80}\n")

# Uncomment the line below to run interactive mode
# interactive_rag()

print("✓ RAG system ready!")
print("To use interactive mode, uncomment and run: interactive_rag()")

✓ RAG system ready!
To use interactive mode, uncomment and run: interactive_rag()
